In [1]:
!pip install timm -q

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms
import timm
from sklearn.metrics import classification_report

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Available GPUs:", torch.cuda.device_count())

base_dir = "/kaggle/input/datasets/abdelghaniaaba/wildfire-prediction-dataset"
train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")

Available GPUs: 2


In [2]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
valid_dataset = datasets.ImageFolder(valid_dir, transform=val_transform)

targets = train_dataset.targets
class_counts = [targets.count(0), targets.count(1)]
class_weights = [sum(class_counts) / (2 * c) for c in class_counts]
sample_weights = [class_weights[label] for _, label in train_dataset]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [3]:
model_vit = timm.create_model('vit_tiny_patch16_224', pretrained=True, num_classes=2)

if torch.cuda.device_count() > 1:
    model_vit = nn.DataParallel(model_vit)

model_vit = model_vit.to(device)

criterion = nn.CrossEntropyLoss()
optimizer_vit = optim.AdamW(model_vit.parameters(), lr=0.0001)

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

In [4]:
epochs = 10

for epoch in range(epochs):
    model_vit.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_vit.zero_grad()
        outputs = model_vit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_vit.step()
        
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    train_acc = correct / total
    print(f"ViT-Tiny Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f}")

torch.save(model_vit.module.state_dict() if isinstance(model_vit, nn.DataParallel) else model_vit.state_dict(), "/kaggle/working/vit_tiny_wildfire.pth")

ViT-Tiny Epoch 1 | Train Loss: 0.0722 | Train Acc: 0.9740
ViT-Tiny Epoch 2 | Train Loss: 0.0286 | Train Acc: 0.9899
ViT-Tiny Epoch 3 | Train Loss: 0.0198 | Train Acc: 0.9929
ViT-Tiny Epoch 4 | Train Loss: 0.0154 | Train Acc: 0.9949
ViT-Tiny Epoch 5 | Train Loss: 0.0157 | Train Acc: 0.9946
ViT-Tiny Epoch 6 | Train Loss: 0.0130 | Train Acc: 0.9956
ViT-Tiny Epoch 7 | Train Loss: 0.0112 | Train Acc: 0.9960
ViT-Tiny Epoch 8 | Train Loss: 0.0092 | Train Acc: 0.9968
ViT-Tiny Epoch 9 | Train Loss: 0.0090 | Train Acc: 0.9968
ViT-Tiny Epoch 10 | Train Loss: 0.0097 | Train Acc: 0.9966


In [5]:
model_deit = timm.create_model('deit_tiny_patch16_224', pretrained=True, num_classes=2)

if torch.cuda.device_count() > 1:
    model_deit = nn.DataParallel(model_deit)

model_deit = model_deit.to(device)
optimizer_deit = optim.AdamW(model_deit.parameters(), lr=0.0001)

model.safetensors: reconstructing file:   0%|          |  0.00B / 22.9MB            

model.safetensors: downloading bytes:           |  0.00B            

In [6]:
for epoch in range(epochs):
    model_deit.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_deit.zero_grad()
        outputs = model_deit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_deit.step()
        
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
    train_acc = correct / total
    print(f"DeiT-Tiny Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.4f}")

model_deit.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in valid_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_deit(images)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

torch.save(model_deit.module.state_dict() if isinstance(model_deit, nn.DataParallel) else model_deit.state_dict(), "/kaggle/working/deit_tiny_wildfire.pth")
print(classification_report(all_labels, all_preds))

DeiT-Tiny Epoch 1 | Train Loss: 0.0701 | Train Acc: 0.9755
DeiT-Tiny Epoch 2 | Train Loss: 0.0309 | Train Acc: 0.9895
DeiT-Tiny Epoch 3 | Train Loss: 0.0212 | Train Acc: 0.9932
DeiT-Tiny Epoch 4 | Train Loss: 0.0176 | Train Acc: 0.9945
DeiT-Tiny Epoch 5 | Train Loss: 0.0161 | Train Acc: 0.9946
DeiT-Tiny Epoch 6 | Train Loss: 0.0131 | Train Acc: 0.9956
DeiT-Tiny Epoch 7 | Train Loss: 0.0108 | Train Acc: 0.9965
DeiT-Tiny Epoch 8 | Train Loss: 0.0114 | Train Acc: 0.9959
DeiT-Tiny Epoch 9 | Train Loss: 0.0082 | Train Acc: 0.9972
DeiT-Tiny Epoch 10 | Train Loss: 0.0076 | Train Acc: 0.9976
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2820
           1       0.99      0.99      0.99      3480

    accuracy                           0.99      6300
   macro avg       0.99      0.99      0.99      6300
weighted avg       0.99      0.99      0.99      6300

